# World Cup Match Predictor — model exploration

A short walkthrough of the pipeline: load data → fit Elo → fit the Poisson goal model → train the ML classifier → make and explain a prediction.

> Run this notebook from the project root so the relative paths resolve.

In [ ]:
import sys
from pathlib import Path

# Make the src/ modules importable.
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd

from data_loader import list_teams, load_matches
from predictor import MatchPredictor

matches = load_matches()
print(f'{len(matches)} matches, {len(list_teams(matches))} teams')
matches.head()

## Train the full predictor

`MatchPredictor.fit()` fits the Elo system, the Poisson goal model and the ML classifier in one call.

In [ ]:
predictor = MatchPredictor().fit()
print('ML backend:', predictor.classifier.backend)

# Top 8 teams by learned Elo rating
elo = pd.Series(predictor.elo.ratings).sort_values(ascending=False)
elo.head(8)

## Inspect attack & defence strengths

Strength of 1.0 means exactly league average. Higher attack = scores more; lower defence = concedes less.

In [ ]:
strength = pd.DataFrame({
    'attack': predictor.poisson.attack,
    'defence': predictor.poisson.defence,
}).sort_values('attack', ascending=False)
strength.round(2).head(10)

## Make a prediction

The result is a dictionary with probabilities, expected goals, likely scorelines, a confidence rating and a plain-English explanation.

In [ ]:
result = predictor.predict('Brazil', 'Germany', stage='Final', neutral=True)
p = result['probabilities']
print(f"Brazil win {p['home_win']*100:.1f}%  |  Draw {p['draw']*100:.1f}%  |  Germany win {p['away_win']*100:.1f}%")
print('Expected goals:', result['expected_goals'])
print('Confidence:', result['confidence'])
print()
print(result['explanation'])

In [ ]:
# Most likely scorelines and key factors
display(pd.DataFrame(result['top_scorelines']))
display(pd.DataFrame(result['key_factors']))

## Simulate a tournament

Monte-Carlo a single-elimination bracket to estimate each team's title chance.

In [ ]:
bracket = ['Brazil', 'France', 'Argentina', 'Spain', 'Germany', 'England', 'Portugal', 'Netherlands']
odds = predictor.simulate_tournament(bracket, n_sims=5000)
pd.Series(odds).sort_values(ascending=False).map(lambda v: f'{v*100:.1f}%')